# Multilingual Health QANotebook 2: Fine-Tuning mt5-small
Multilingual Health Question Answering in Low-Resource African Languages (Zindi)  

## Experiment Log
|  | Description | Zindi Score |
|---|---|---|
| 1 | Zero-shot mt5-small (no fine-tuning) | 0.000926 |
| 2 | Fine-tuned, 1 epoch, lr=3e-4 | 0.232357 |
| 3 | Fine-tuned, 3 epochs, lr=3e-4 | 0.235209 |
| 4 | Fine-tuned, 3 epochs, lr=1e-4 | 0.226766 |
| 5 | Fine-tuned, Train+Val combined, 3 epochs, lr=3e-4 | 0.261357 |
| 6 | Fine-tuned, Train+Val combined, 5 epochs, beams=4 | 0.177526 |
| 7 | Fine-tuned, Train+Val combined, 5 epochs, greedy decoding | 0.179376 |
| 8 | Fine-tuned, Train+Val combined, 5 epochs, 512 max tokens | 0.177489 |
| 9 | Fine-tuned, Train+Val combined, 3 epochs, warmup=500, beams=4 | 0.262992 |
| 10 | Fine-tuned, Train+Val combined, 3 epochs, warmup=500, beams=6 | **0.266956 (best)** |

**Key insights:**
- Finetuning produces a 250x improvement over zero-shot (Exp 1 to Exp 2)
- Combining Train+Val data was the single biggest improvement after initial fine-tuning (Exp 3 + Exp 5)
- 5 epochs consistently overfits relative to 3 epochs, training loss kept dropping (down to 43.49)
  but Zindi score dropped sharply from 0.261 to ~0.18 (Exp 5 vs Exp 6-8), a clear bias-variance tradeoff
- A longer warmup (500 steps) and wider beam search (6 beams) gave the best final result,
  showing that careful inference-time and optimizer settings still matter after the main
  architecture/data decisions are fixed


## 0. Setup

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece accelerate -q
print('Done!')

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Load & Preprocess Data

In [ ]:
DATA_PATH = '/content/drive/MyDrive/health_qa_data/'
SUBMISSION_PATH = DATA_PATH + 'submissions/'
MODEL_SAVE_PATH = DATA_PATH + 'models/'
os.makedirs(SUBMISSION_PATH, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

train_df = pd.read_csv(DATA_PATH + 'Train.csv')
val_df   = pd.read_csv(DATA_PATH + 'Val.csv')
test_df  = pd.read_csv(DATA_PATH + 'Test.csv')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:

subset_to_prefix = {
    'Aka_Gha':  'answer in Akan: ',
    'Amh_Eth':  'answer in Amharic: ',
    'Eng_Eth':  'answer in English: ',
    'Eng_Gha':  'answer in English: ',
    'Eng_Ken':  'answer in English: ',
    'Eng_Uga':  'answer in English: ',
    'Lug_Uga':  'answer in Luganda: ',
    'Swa_Ken':  'answer in Kiswahili: ',
}

def preprocess_df(df, is_test=False):
    df = df.copy()
    df['input'] = df['input'].str.strip()
    if not is_test:
        df['output'] = df['output'].str.strip()
        df = df.dropna(subset=['input', 'output'])
    else:
        df = df.dropna(subset=['input'])
    df['model_input'] = df['subset'].map(subset_to_prefix) + df['input']
    return df

train_clean = preprocess_df(train_df)
val_clean   = preprocess_df(val_df)
test_clean  = preprocess_df(test_df, is_test=True)

print(f'Train: {len(train_clean):,} | Val: {len(val_clean):,} | Test: {len(test_clean):,}')
print('Sample input:', train_clean['model_input'].iloc[0][:100])

## 2. Tokenize Dataset

The tokenizer converts text to numbers and back.

**Max lengths were chosen based on EDA:**
- Input: 256 tokens (covers 95% and above of questions)
- Output: 256 tokens (covers most answers)

In [ ]:
MODEL_NAME = 'google/mt5-small'
MAX_INPUT_LENGTH  = 256
MAX_OUTPUT_LENGTH = 256

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

In [ ]:
def tokenize_function(examples):
    """
    Tokenize input questions and output answers.
    Labels use -100 for padding so the model ignores padding during loss computation.
    """
    model_inputs = tokenizer(
        examples['model_input'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )
    
    labels = tokenizer(
        examples['output'],
        max_length=MAX_OUTPUT_LENGTH,
        truncation=True,
        padding=False
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Convert to HuggingFace Dataset format
print('Converting to HuggingFace Dataset format...')
train_dataset = Dataset.from_pandas(train_clean[['model_input', 'output']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_clean[['model_input', 'output']].reset_index(drop=True))

print('Tokenizing train dataset...')
train_tokenized = train_dataset.map(tokenize_function, batched=True, batch_size=1000,
                                     remove_columns=['model_input', 'output'])

print('Tokenizing val dataset...')
val_tokenized = val_dataset.map(tokenize_function, batched=True, batch_size=1000,
                                 remove_columns=['model_input', 'output'])

print(f'Train tokenized: {len(train_tokenized):,} examples')
print(f'Val tokenized:   {len(val_tokenized):,} examples')

## 3. Experiment 2 Fine-tune mt5-small, 1 Epoch

In [ ]:
# EXPERIMENT 2 CONFIG change these to run different experiments later

EXPERIMENT_NUM   = 2
NUM_EPOCHS       = 1        
LEARNING_RATE    = 5e-4     
BATCH_SIZE       = 8        
GRAD_ACCUM       = 4        
WARMUP_STEPS     = 200

print(f'=== EXPERIMENT {EXPERIMENT_NUM} CONFIG ===')
print(f'Epochs:        {NUM_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Batch size:    {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})')
print(f'Warmup steps:  {WARMUP_STEPS}')

In [ ]:
# Load fresh model for fine-tuning
print('Loading model...')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)
print(f'Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Data collator  
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

# Training arguments
output_dir = f'/content/drive/MyDrive/health_qa_data/models/exp{EXPERIMENT_NUM}'

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=256,
    logging_steps=100,
    fp16=False,                   
    report_to='none',           
    save_total_limit=1,           
)

print('Training arguments configured!')

In [ ]:
# ROUGE metric for evaluation during training
rouge = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    vocab_size = len(tokenizer)
    preds = np.clip(preds, 0, vocab_size - 1)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, vocab_size - 1)
    
    decoded_preds  = tokenizer.batch_decode(preds,   skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )
    
    return {
        'rouge1': round(result['rouge1'], 4),
        'rougeL': round(result['rougeL'], 4)
    }

print('Metrics function defined!')

In [ ]:
# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('Trainer initialized!')
print()
print('=== STARTING TRAINING ===')
print(f'Training on {len(train_tokenized):,} examples for {NUM_EPOCHS} epoch(s)')
print('This will take approximately 45-60 minutes on free Colab T4...')
print('DO NOT close the browser tab!')

In [ ]:
# Track training loss for plotting
train_losses = []

# START TRAINING
train_result = trainer.train()

print()
print('=== TRAINING COMPLETE ===')
print(f'Training loss: {train_result.training_loss:.4f}')
print(f'Total steps:   {train_result.global_step}')
print(f'Time taken:    {train_result.metrics["train_runtime"] / 60:.1f} minutes')

In [ ]:
# Plot training loss curve
log_history = trainer.state.log_history
train_logs = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if train_logs:
    steps  = [x['step'] for x in train_logs]
    losses = [x['loss'] for x in train_logs]
    axes[0].plot(steps, losses, color='steelblue', linewidth=1.5)
    axes[0].set_title(f'Exp {EXPERIMENT_NUM} — Training Loss', fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

if eval_logs:
    eval_steps  = [x['step'] for x in eval_logs]
    eval_r1     = [x.get('eval_rouge1', 0) for x in eval_logs]
    eval_rl     = [x.get('eval_rougeL', 0) for x in eval_logs]
    axes[1].plot(eval_steps, eval_r1, label='ROUGE-1', color='coral', linewidth=2)
    axes[1].plot(eval_steps, eval_rl, label='ROUGE-L', color='steelblue', linewidth=2)
    axes[1].set_title(f'Exp {EXPERIMENT_NUM} — Validation ROUGE', fontweight='bold')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('ROUGE Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/health_qa_data/exp{EXPERIMENT_NUM}_learning_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Learning curve saved!')

In [ ]:
# Evaluate on validation set
print('Running final evaluation on validation set...')
eval_results = trainer.evaluate()
print()
print(f'=== EXPERIMENT {EXPERIMENT_NUM} VALIDATION RESULTS ===')
print(f'Val ROUGE-1: {eval_results.get("eval_rouge1", "N/A")}')
print(f'Val ROUGE-L: {eval_results.get("eval_rougeL", "N/A")}')
print(f'Val Loss:    {eval_results.get("eval_loss", "N/A"):.4f}')

## 4. Generate Predictions & Submit

In [ ]:
# Generation function
def generate_answers(texts, model, tokenizer, batch_size=16,
                     max_input_length=256, max_output_length=256):
    model.eval()
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            max_length=max_input_length,
            truncation=True,
            padding=True
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_output_length,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_outputs.extend(decoded)
        if (i // batch_size) % 10 == 0:
            print(f'  {i+len(batch)}/{len(texts)} done...')
    return all_outputs

print(f'Generating predictions for {len(test_clean)} test examples...')
test_inputs = test_clean['model_input'].tolist()
test_preds  = generate_answers(test_inputs, model, tokenizer, batch_size=16)
print('Done!')

In [ ]:
# Build and save submission
submission = pd.DataFrame({
    'ID':         test_clean['ID'].tolist(),
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds
})

submission_file = SUBMISSION_PATH + f'submission_exp{EXPERIMENT_NUM}_finetuned_mt5small_{NUM_EPOCHS}ep_lr{LEARNING_RATE}.csv'
submission.to_csv(submission_file, index=False)

print(f'Submission saved: {submission_file}')
print()
print('Preview:')
print(submission.head(3))
print()
print('NEXT: Download this CSV from Drive and submit to Zindi!')

## 5. Analysis & Experiment Notes (Experiments 2 to 10)

**Experiment 2 ; Fine-tuned mt5-small, 1 epoch, lr=3e-4**
- What changed from Exp 1: Added fine-tuning on 29,815 training examples
- Why: Zero-shot baseline scored 0.000926, the model needs task-specific training
- Zindi score: 0.232357
- Insight: Even one epoch of fine-tuning produces a 250x improvement over zero-shot. The model goes from outputting `<extra_id_0>` template tokens to coherent, language-appropriate health answers in Akan, Luganda, Kiswahili, Amharic, and English.

**Experiment 3 ; Fine-tuned mt5-small, 3 epochs, lr=3e-4**
- What changed: Increased training from 1 to 3 epochs
- Why: More training should let the model learn more from the same data
- Zindi score: 0.235209
- Insight: The improvement from 1 to 3 epochs (0.232 to 0.235) is far smaller than 0 to 1 epoch (0.000926 to 0.232). Diminishing returns set in quickly with this small model and dataset size.

**Experiment 4 ; Fine-tuned mt5-small, 3 epochs, lr=1e-4**
- What changed: Lowered learning rate from 3e-4 to 1e-4
- Why: Test whether a smaller, more careful learning rate improves generalization
- Zindi score: 0.226766
- Insight: Despite reaching a lower training loss (65.67) than Exp 3 (122.58), Exp 4 scored lower on the leaderboard. Lower training loss does not guarantee better generalization , a concrete example of the gap between fitting training data and performing well on unseen test data.

**Experiment 5 ; Fine-tuned mt5-small, Train+Val combined, 3 epochs, lr=3e-4**
- What changed: Combined Train.csv and Val.csv into one training set (36,501 examples instead of 29,815)
- Why: More training data generally improves generalization
- Zindi score: 0.261357
- Insight: A 22% increase in training data produced the largest single improvement after the initial fine-tuning step itself. Data quantity mattered more than any individual hyperparameter change tried so far.

**Experiment 6 ; Train+Val combined, 5 epochs, lr=3e-4, beam search (beams=4)**
- What changed: Increased epochs from 3 to 5
- Why: Test whether more training continues to help once Train+Val data is used
- Zindi score: 0.177526
- Insight: Training loss kept falling (43.49 final loss, the lowest yet at that point) but the Zindi score dropped sharply from 0.261 to 0.178, a clear case of overfitting: the model memorized training-set patterns that did not transfer to the held-out test set.

**Experiment 7 ; Train+Val combined, 5 epochs, lr=3e-4, greedy decoding (beams=1)**
- What changed: Used the same overfit 5-epoch model from Exp 6, switched decoding from beam search to greedy
- Why: Isolate whether the Exp 6 drop was caused by overfitting or by decoding strategy
- Zindi score: 0.179376
- Insight: Score stayed essentially unchanged from Exp 6, confirming the drop was caused by overfitting during training, not by the choice of decoding strategy.

**Experiment 8 ; Train+Val combined, 5 epochs, lr=3e-4, longer output (512 tokens)**
- What changed: Same overfit model, increased max_new_tokens from 256 to 512
- Why: Test whether answers were being truncated and a longer output budget would recover lost score
- Zindi score: 0.177489
- Insight: Again essentially unchanged from Exp 6/7, confirming output length was not the limiting factor, the overfit model weights themselves were the problem, not generation settings.

**Experiment 9 ; Train+Val combined, 3 epochs, lr=3e-4, warmup_steps=500, beams=4**
- What changed: Returned to 3 epochs (the configuration that worked best in Exp 5), increased warmup_steps from 200 to 500, and used a larger effective batch size (4 × 8 grad-accum = 32) for more stable gradients
- Why: Having confirmed 5 epochs overfits, this tests whether a more gradual learning-rate warmup improves on the Exp 5 baseline
- Zindi score: 0.262992
- Insight: A small but real improvement over Exp 5 (0.261357 to 0.262992). A longer warmup lets the optimizer take smaller, more careful steps early in training, slightly improving the final solution without changing epochs or learning rate.

**Experiment 10 ; Train+Val combined, 3 epochs, lr=3e-4, warmup_steps=500, beams=6**
- What changed: Same trained model as Exp 9, increased beam search width from 4 to 6 at inference time
- Why: A wider beam search explores more candidate sequences during generation and can surface a better final answer, at the cost of slower inference
- Zindi score: **0.266956 to best result across all 10 experiments**
- Insight: Widening the beam search gave the largest gain of any inference-only change (no retraining needed), improving on Exp 9 by 0.004. Combined with the Exp 5 to 9 data/warmup gains, this represents a ~14% relative improvement in Zindi score over the first fine-tuned model (Exp 2: 0.2324 to Exp 10: 0.2670), achieved entirely through systematic, documented experimentation rather than a single lucky configuration.


### Overall takeaways
1. **Fine-tuning is non-negotiable**: zero-shot mT5 cannot do this task (Exp 1).
2. **Data quantity beat most hyperparameter tweaks** : combining Train+Val (Exp 5) was the single largest gain after the initial fine-tuning step.
3. **More epochs is not always better** : 5 epochs clearly overfit relative to 3 (Exp 5 vs Exp 6-8), and training loss alone is a misleading signal of test-set performance (Exp 4 also illustrates this).
4. **Inference-time settings still matter at the margins** : warmup steps and beam width gave the final, best-scoring configuration (Exp 9-10) without any new data or epochs.


## 6. Final Pipeline — Reproducing the Best Model (Experiments 9 & 10)

Experiments 3-8 followed the same structure as Experiment 2 above, changing only the
`EXPERIMENT_NUM`, `NUM_EPOCHS`, `LEARNING_RATE`, and (from Experiment 5 onward) combining
`Train.csv` with `Val.csv` before tokenizing. Their results are summarized in Section 5 above.

The cell below is a single, self-contained pipeline that reproduces the **best two
configurations found** — Experiments 9 and 10 — which share the same trained model and
differ only in beam width at inference time. Run it after running Sections 0–1 above
(setup, data loading, and the `subset_to_prefix`/`preprocess` definitions).

In [ ]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
assert device == 'cuda', 'No GPU detected — go to Runtime > Change runtime type > T4 GPU'

DATA_PATH = '/content/drive/MyDrive/health_qa_data/'
SUBMISSION_PATH = DATA_PATH + 'submissions/'
os.makedirs(SUBMISSION_PATH, exist_ok=True)

train_df = pd.read_csv(DATA_PATH + 'Train.csv')
val_df   = pd.read_csv(DATA_PATH + 'Val.csv')
test_df  = pd.read_csv(DATA_PATH + 'Test.csv')

# Experiment 5+ insight: combining Train+Val gives the biggest single improvement
train_df = pd.concat([train_df, val_df]).reset_index(drop=True)
print(f'Combined train size: {len(train_df):,}')

def preprocess(df, is_test=False):
    df = df.copy()
    df['input'] = df['input'].str.strip()
    if not is_test:
        df['output'] = df['output'].str.strip()
        df = df.dropna(subset=['input', 'output'])
    else:
        df = df.dropna(subset=['input'])
    df['model_input'] = df['subset'].map(subset_to_prefix) + df['input']
    return df.reset_index(drop=True)

train_clean = preprocess(train_df)
test_clean  = preprocess(test_df, is_test=True)
print(f'Train: {len(train_clean):,} | Test: {len(test_clean):,}')

def tokenize_function_full(examples):
    model_inputs = tokenizer(examples['model_input'], max_length=MAX_INPUT_LENGTH,
                              truncation=True, padding=False)
    labels = tokenizer(examples['output'], max_length=MAX_OUTPUT_LENGTH,
                        truncation=True, padding=False)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset = Dataset.from_pandas(train_clean[['model_input', 'output']])
train_tokenized = train_dataset.map(tokenize_function_full, batched=True, batch_size=1000,
                                     remove_columns=['model_input', 'output'])
print('Tokenization done!')

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model,
                                        padding=True, label_pad_token_id=-100)


training_args_final = Seq2SeqTrainingArguments(
    output_dir='/content/model_exp9_10',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,   
    learning_rate=3e-4,
    warmup_steps=500,                
    weight_decay=0.01,
    eval_strategy='no',
    save_strategy='no',
    predict_with_generate=False,
    logging_steps=100,
    fp16=False,
    report_to='none',
)

trainer_final = Seq2SeqTrainer(
    model=model,
    args=training_args_final,
    train_dataset=train_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print('Starting Exp 9/10 base model — Train+Val, 3 epochs, warmup=500...')
train_result_final = trainer_final.train()
print(f'Training complete! Final training loss: {train_result_final.training_loss:.4f}')

def generate_answers_final(texts, model, tokenizer, batch_size=8, max_tokens=256, beams=4):
    model.eval()
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', max_length=256,
                            truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_tokens,
                                      num_beams=beams, no_repeat_ngram_size=3)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_outputs.extend(decoded)
        if (i // batch_size) % 10 == 0:
            print(f'  {i+len(batch)}/{len(texts)} done...')
    return all_outputs

test_inputs = test_clean['model_input'].tolist()
test_ids = test_clean['ID'].tolist()

print('Generating Exp 9 (beams=4)...')
preds9 = generate_answers_final(test_inputs, model, tokenizer, beams=4, max_tokens=256)
pd.DataFrame({
    'ID': test_ids, 'TargetRLF1': preds9, 'TargetR1F1': preds9, 'TargetLLM': preds9
}).to_csv(SUBMISSION_PATH + 'submission_exp9_warmup500.csv', index=False)
print('Exp 9 saved to Drive!')


print('Generating Exp 10 (beams=6)...')
preds10 = generate_answers_final(test_inputs, model, tokenizer, beams=6, max_tokens=256)
pd.DataFrame({
    'ID': test_ids, 'TargetRLF1': preds10, 'TargetR1F1': preds10, 'TargetLLM': preds10
}).to_csv(SUBMISSION_PATH + 'submission_exp10_beams6.csv', index=False)
print('Exp 10 saved to Drive!')

print()
print('ALL DONE! Experiment 10 is the final, best-performing model (Zindi score: 0.266956).')
print('Both submissions saved to:', SUBMISSION_PATH)
